In [1]:
from google.colab import userdata
import os

os.environ["GEMINI_API_KEY"] = userdata.get("GEMINI_API_KEY")

In [2]:
print("API key loaded:", bool(os.environ.get("GEMINI_API_KEY")))

API key loaded: True


In [ ]:
!pip install -q youtube-transcript-api langchain-community langchain-google-genai faiss-cpu tiktoken python-dotenv

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.
google-colab 1.0.0 requires requests==2.32.4, but you have requests 2.34.2 which is incompatible.


In [ ]:
!pip install -q requests==2.32.4 google-auth==2.62.0

ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
langchain-community 0.4.2 requires requests<3.0.0,>=2.32.5, but you have requests 2.32.4 which is incompatible.
google-colab 1.0.0 requires google-auth==2.49.0, but you have google-auth 2.62.0 which is incompatible.


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

from langchain_text_splitters import RecursiveCharacterTextSplitter

from langchain_google_genai import (
    GoogleGenerativeAIEmbeddings,
    ChatGoogleGenerativeAI
)

from langchain_community.vectorstores import FAISS

from langchain_core.prompts import PromptTemplate

Step 1a : Indexing(Document ingestion)


In [ ]:
from youtube_transcript_api import YouTubeTranscriptApi, TranscriptsDisabled

video_id = "jNQXAC9IVRw"  # Only video ID, not full URL

try:
    # Create API instance and fetch transcript
    ytt_api = YouTubeTranscriptApi()
    transcript_list = ytt_api.fetch(
        video_id,
        languages=["en"]
    )

    # Convert transcript snippets into plain text
    transcript = " ".join(
        chunk.text for chunk in transcript_list
    )

    print(transcript)

except TranscriptsDisabled:
    print("No captions available for this video.")

except Exception as e:
    print("Error:", e)

Error: 
Could not retrieve a transcript for the video https://www.youtube.com/watch?v=jNQXAC9IVRw! This is most likely caused by:

YouTube is blocking requests from your IP. This usually is due to one of the following reasons:
- You have done too many requests and your IP has been blocked by YouTube
- You are doing requests from an IP belonging to a cloud provider (like AWS, Google Cloud Platform, Azure, etc.). Unfortunately, most IPs from cloud providers are blocked by YouTube.

There are two things you can do to work around this:
1. Use proxies to hide your IP address, as explained in the "Working around IP bans" section of the README (https://github.com/jdepoix/youtube-transcript-api?tab=readme-ov-file#working-around-ip-bans-requestblocked-or-ipblocked-exception).
2. (NOT RECOMMENDED) If you authenticate your requests using cookies, you will be able to continue doing requests for a while. However, YouTube will eventually permanently ban the account that you have used to authenticate

In [ ]:
with open("/content/transcript.txt", "r", encoding="utf-8") as f:
    transcript = f.read()

Step 1b: Indexing (Text splitting)

In [ ]:
splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
chunks = splitter.create_documents([transcript])

In [ ]:
len(chunks)

12

Step 1c & 1d - Indexing (Embedding Generation and Storing in Vector Store)

In [ ]:
embeddings = GoogleGenerativeAIEmbeddings(
    model="models/gemini-embedding-001",
    google_api_key=os.environ["GEMINI_API_KEY"]
)

In [ ]:
import time
from langchain_community.vectorstores import FAISS

batch_size = 20
vector_store = None

for i in range(0, len(chunks), batch_size):
    batch = chunks[i:i + batch_size]

    print(f"Embedding chunks {i + 1} to {i + len(batch)}...")

    if vector_store is None:
        vector_store = FAISS.from_documents(batch, embeddings)
    else:
        vector_store.add_documents(batch)

    print("Batch completed!")

    if i + batch_size < len(chunks):
        print("Waiting 65 seconds to avoid rate limits...")
        time.sleep(65)

print("All chunks embedded successfully!")

Embedding chunks 1 to 12...
Batch completed!
All chunks embedded successfully!


In [ ]:
vector_store.index_to_docstore_id

{0: '94563584-00b2-40e7-8fd4-6eaa5be8ba1e',
 1: '1bced29f-0ce1-43ee-b7d4-bce03a4cdeae',
 2: 'f8d7c882-53de-4a8b-b247-01b35daec5a1',
 3: '65add386-1153-48d2-acd9-74dd6482d919',
 4: '819d9bdb-f326-4269-bf77-1f6673863e00',
 5: '4712840e-2b1b-4fea-aac6-6c4a9d481c36',
 6: '7cb03f34-4d96-4a72-afa7-b4a540126d93',
 7: 'ef66a418-2937-4f19-a902-dc8ad0712309',
 8: 'b03bcad6-c220-4fba-a686-e19963fc9bac',
 9: '1cf93a3d-03e9-499a-b8ed-57d74b4fca99',
 10: '666c9496-8694-4410-8aff-e71f59885a24',
 11: 'bed890d1-f13f-421b-a398-c6871d07e470'}

Step 2: Retrieval

In [ ]:
retriever = vector_store.as_retriever(search_type="similarity", search_kwargs={"k": 4})

In [ ]:
retriever

VectorStoreRetriever(tags=['FAISS', 'GoogleGenerativeAIEmbeddings'], vectorstore=<langchain_community.vectorstores.faiss.FAISS object at 0x7f179420acf0>, search_kwargs={'k': 4})

In [ ]:
query = "What is this video about?"

docs = retriever.invoke(query)

for i, doc in enumerate(docs, start=1):
    print(f"\n--- Document {i} ---")
    print(doc.page_content)



--- Document 1 ---
everybody's talking about artificial intelligence these days AI machine learning is another Hot Topic are they the same thing or are they different and if so what are those differences and deep learning is another one that comes into play I actually did a video on these three artificial intelligence machine learning and deep learning and talked about where they fit and there were a lot of comments on that and I read those comments and I'd like to address some of the most frequently asked questions so that we clear up some of the myths and misconceptions around this in addition something else has happened since that video was recorded and that is this the absolute explosion of this area of generative AI things like large language models and chat Bots have seemed to be taking over the world we see them everywhere really interesting technology uh and then also things like deep fakes these are all within the realm of AI but how do they fit within each other how are they

Step 3: Augmentation

In [ ]:
llm = ChatGoogleGenerativeAI(
    model="gemini-3.6-flash",
    temperature=0.2,
    google_api_key=os.environ["GEMINI_API_KEY"]
)

In [ ]:
prompt = PromptTemplate(
    template="""
      You are a helpful assistant.
Answer ONLY using the provided transcript context.
If the context is insufficient, say "I don't know."

Give a clear, detailed answer in numbered points.
Include the key details relevant to the question.

Context:
{context}

Question: {question}

Answer:
""",
    input_variables=["context", "question"]
)

In [ ]:
question = "What is the main topic discussed in this video?"
retrieved_docs = retriever.invoke(question)

In [ ]:
retrieved_docs


[Document(id='94563584-00b2-40e7-8fd4-6eaa5be8ba1e', metadata={}, page_content="everybody's talking about artificial intelligence these days AI machine learning is another Hot Topic are they the same thing or are they different and if so what are those differences and deep learning is another one that comes into play I actually did a video on these three artificial intelligence machine learning and deep learning and talked about where they fit and there were a lot of comments on that and I read those comments and I'd like to address some of the most frequently asked questions so that we clear up some of the myths and misconceptions around this in addition something else has happened since that video was recorded and that is this the absolute explosion of this area of generative AI things like large language models and chat Bots have seemed to be taking over the world we see them everywhere really interesting technology uh and then also things like deep fakes these are all within the re

In [ ]:
context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
context_text

"everybody's talking about artificial intelligence these days AI machine learning is another Hot Topic are they the same thing or are they different and if so what are those differences and deep learning is another one that comes into play I actually did a video on these three artificial intelligence machine learning and deep learning and talked about where they fit and there were a lot of comments on that and I read those comments and I'd like to address some of the most frequently asked questions so that we clear up some of the myths and misconceptions around this in addition something else has happened since that video was recorded and that is this the absolute explosion of this area of generative AI things like large language models and chat Bots have seemed to be taking over the world we see them everywhere really interesting technology uh and then also things like deep fakes these are all within the realm of AI but how do they fit within each other how are they related to each\n\

In [ ]:
final_prompt = prompt.invoke({"context": context_text, "question": question})

In [ ]:
final_prompt

StringPromptValue(text='\n      You are a helpful assistant.\nAnswer ONLY using the provided transcript context.\nIf the context is insufficient, say "I don\'t know."\n\nGive a clear, detailed answer in numbered points.\nInclude the key details relevant to the question.\n\nContext:\neverybody\'s talking about artificial intelligence these days AI machine learning is another Hot Topic are they the same thing or are they different and if so what are those differences and deep learning is another one that comes into play I actually did a video on these three artificial intelligence machine learning and deep learning and talked about where they fit and there were a lot of comments on that and I read those comments and I\'d like to address some of the most frequently asked questions so that we clear up some of the myths and misconceptions around this in addition something else has happened since that video was recorded and that is this the absolute explosion of this area of generative AI th

Step 4 : Generation

In [ ]:
import re

answer = llm.invoke(final_prompt)
text = answer.content

if isinstance(text, list):
    text = "\n".join(
        item["text"] for item in text
        if isinstance(item, dict) and "text" in item
    )

text = re.sub(r"\s+(\d+\.)", r"\n\1", text)
text = re.sub(r"'signature':.*", "", text, flags=re.DOTALL)

print(text.strip())

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


Based on the provided transcript, the main topic discussed in the video includes:
1. **How AI technologies relate to each other:** Explaining how Artificial Intelligence (AI), Machine Learning, Deep Learning, and Generative AI (including chatbots, large language models, and deep fakes) fit together, relate to one another, and can be used.
2. **Clearing up myths and misconceptions:** Addressing frequently asked questions to clarify whether these topics are the same or different.
3. **Defining key AI concepts:** Defining what AI is (simulating/matching human intelligence to learn, infer, and reason) and what Machine Learning is (allowing a machine to learn from information and observations without direct programming).
4. **The evolution of AI adoption:** Explaining how the adoption of AI shifted from slow early adoption to rapid, widespread growth ("going straight to the Moon") due to the rise of foundation models and Generative AI.


In [ ]:
from langchain_core.runnables import RunnableParallel, RunnablePassthrough, RunnableLambda
from langchain_core.output_parsers import StrOutputParser

In [ ]:
def format_docs(retrieved_docs):
  context_text = "\n\n".join(doc.page_content for doc in retrieved_docs)
  return context_text

In [ ]:
parallel_chain = RunnableParallel({
    'context': retriever | RunnableLambda(format_docs),
    'question': RunnablePassthrough()
})

In [ ]:
parallel_chain.invoke("What is Generative AI?")

{'context': "a little bit like the autoc complete when you start typing something in and then it predicts what your next word will be except in this case with large language models they're not predicting the next word they're predicting the next sentence the next paragraph the next entire document so there's a really an amazing exponential leap in what these things are able to do and we call all of these Technologies generative because they are generating new content um some people have actually made the argument that the generative AI isn't really generative that that these Technologies are really just regurgitating existing information and putting it in different format well let me give you an analogy um if you take music for instance then every note has already been invented so in a sense every song is just a recombination some other permutation of all the notes that already exist already and just putting them in a different order well we don't say new new music doesn't exist people

In [ ]:
parser = StrOutputParser()

In [ ]:
main_chain = parallel_chain | prompt | llm | parser

In [ ]:
main_chain.invoke('Can you summarize the video')

/usr/local/lib/python3.13/dist-packages/langchain_google_genai/chat_models.py:3919: UserWarning: Model 'gemini-3.6-flash' uses fixed sampling defaults; the sampling parameter(s) temperature will be ignored.
  request = self._build_request_config(


'Based on the provided transcript, here is a summary of the video:\n\n1. **Definition of AI:** Artificial Intelligence (AI) is defined as trying to use a computer to simulate, match, or exceed human intelligence, which involves abilities such as learning, inferring, and reasoning.\n2. **Purpose of the Video:** The speaker aims to simplify and explain how various related technologies—including machine learning, deep learning, generative AI, foundation models, chatbots, and deep fakes—fit together, while clearing up common myths and FAQs from a previous video.\n3. **The Rise of Generative AI and Foundation Models:** Since the speaker\'s last video, there has been an explosion of generative AI, large language models (LLMs), and foundation models. These technologies are designed to generate new content or summarize existing content into bite-sized, manageable formats.\n4. **Use Cases and Potential Abuse:** Technologies stemming from generative AI and foundation models (like chatbots and de